# 01 数据理解：Scania APS 预测性维护项目\n\n本 notebook 用于 Day 1 的项目初始化和数据理解准备。当前阶段只确认业务背景、数据路径、基础读取方式和标签含义，不进行复杂清洗、特征工程、模型训练或阈值调优。\n\n业务成本设定：误报 FP 成本为 10，漏报 FN 成本为 500。后续评估不会以 accuracy 作为核心目标，而会重点关注 precision、recall、F1、F2、PR-AUC 和 total cost。

## 1. 导入依赖\n\n这里只导入 Day 1 数据读取和基础查看需要的依赖。

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\npd.set_option("display.max_columns", 20)\npd.set_option("display.width", 120)

## 2. 设置数据路径\n\n原始数据保存在 `data/raw/`。Day 1 只读取原始 train/test 文件，不修改原始数据，也不合并后重新切分。

In [ ]:
# 支持从项目根目录或 notebooks/ 目录启动 notebook。\nPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()\n\nTRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "aps_failure_training_set.csv"\nTEST_PATH = PROJECT_ROOT / "data" / "raw" / "aps_failure_test_set.csv"\n\nTRAIN_PATH, TEST_PATH

## 3. 读取 train/test 数据\n\n原始 CSV 中的字符串 `na` 表示缺失值，读取时需要通过 `na_values="na"` 转换为 pandas 缺失值。

In [ ]:
train_df = pd.read_csv(TRAIN_PATH, na_values="na")\ntest_df = pd.read_csv(TEST_PATH, na_values="na")\n\ntrain_df.head()

## 4. 查看数据规模\n\n这里只查看行数和列数，不对数据作清洗决策。

In [ ]:
shape_summary = pd.DataFrame(\n    {\n        "数据集": ["train", "test"],\n        "行数": [train_df.shape[0], test_df.shape[0]],\n        "列数": [train_df.shape[1], test_df.shape[1]],\n    }\n)\n\nshape_summary

## 5. 查看字段信息\n\n字段名经过匿名化处理，Day 1 先确认字段数量、字段类型和标签列位置。

In [ ]:
train_df.info()

In [ ]:
train_df.columns[:10].tolist(), train_df.columns[-10:].tolist()

## 6. 查看 class 标签分布\n\n`pos` 表示 APS 系统相关故障，`neg` 表示非 APS 系统相关故障。这里先查看分布，为后续类别不平衡分析做准备。

In [ ]:
label_distribution = (\n    train_df["class"]\n    .value_counts(dropna=False)\n    .rename_axis("class")\n    .reset_index(name="样本数")\n)\n\nlabel_distribution["占比"] = label_distribution["样本数"] / len(train_df)\nlabel_distribution

## 7. 将 class 映射为 target\n\n后续建模会使用 `target`：`pos -> 1`，`neg -> 0`。Day 1 只创建映射列用于理解标签，不做模型训练。

In [ ]:
target_mapping = {"neg": 0, "pos": 1}\n\ntrain_df = train_df.assign(target=train_df["class"].map(target_mapping))\ntest_df = test_df.assign(target=test_df["class"].map(target_mapping))\n\ntrain_df[["class", "target"]].head()

In [ ]:
train_df["target"].value_counts(dropna=False).sort_index()

## Day 1 小结\n\n- 已明确项目业务目标：预测 APS 系统相关故障，并服务维修优先级决策。\n- 已明确标签映射：`pos -> 1`，`neg -> 0`。\n- 已明确成本设定：FP = 10，FN = 500。\n- 已确认原始数据读取方式：`pd.read_csv(..., na_values="na")`。\n- 当前阶段不进行模型训练、复杂清洗、特征工程或阈值选择。